In [2]:
import pandas as pd
import glob
import requests
import time

# Data Loading

In [8]:
all_dfs = []
for filename in glob.glob("../data/*.csv"):
    new_df = pd.read_csv(filename)
    all_dfs.append(new_df)
df = pd.concat(all_dfs)

In [9]:
df = df[df['ORIGIN'].notna()]

In [10]:
df.columns

Index(['FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN',
       'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST',
       'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME',
       'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'ARR_DELAY', 'ARR_DEL15',
       'CANCELLED', 'DIVERTED', 'CRS_ELAPSED_TIME', 'DISTANCE',
       'DISTANCE_GROUP', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
       'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY'],
      dtype='object')

In [11]:
df.to_parquet("../data/raw_data.parquet", index=False)

In [8]:
real_df = pd.read_parquet("../data/raw_data.parquet")

In [9]:
airport_data = pd.read_csv('../airports/airports.csv')
airport_data = airport_data[['iata_code', 'latitude_deg', 'longitude_deg']]
airport_data = airport_data.dropna(subset=['iata_code'])
airport_data = airport_data[airport_data['iata_code'] != 'None']
airport_data = airport_data[airport_data['iata_code'].isin(real_df['ORIGIN'].unique())]

In [17]:
airport_data[['latitude_deg', 'longitude_deg']]

,latitude_deg,longitude_deg
38513,40.651773,-75.442797
38514,32.411301,-99.681900
38515,35.039976,-106.608925
38516,45.449100,-98.421799
38517,31.532946,-84.196215
...,...,...
65121,18.337091,-64.977251
65122,17.701413,-64.802608
65138,18.494900,-67.129402
65143,18.008301,-66.563004


In [10]:
from tqdm import tqdm

In [ ]:
first_batch = airport_data.iloc[:180]

weather_data = []

for _, row in tqdm(first_batch.iterrows(), total=len(first_batch)):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={row['latitude_deg']}&"
        f"longitude={row['longitude_deg']}&"
        f"start_date=2025-01-01&"
        f"end_date=2026-04-30&"
        f"hourly=precipitation,wind_speed_10m,snowfall,temperature_2m,rain,wind_gusts_10m,weather_code,cloud_cover_low"
    )
    
    try:
        response = requests.get(url, timeout=30).json()
        hourly = response['hourly']
        
        response_df = pd.DataFrame({
            'iata_code': row['iata_code'],
            'datetime': hourly['time'],
            'precipitation': hourly['precipitation'],
            'rain': hourly['rain'],
            'snowfall': hourly['snowfall'],
            'wind_speed': hourly['wind_speed_10m'],
            'wind_gusts': hourly['wind_gusts_10m'],
            'temperature': hourly['temperature_2m'],
            'weather_code': hourly['weather_code'],
            'cloud_cover_low': hourly['cloud_cover_low']
        })
        
        weather_data.append(response_df)
    except Exception as e:
        print(f"Failed for {row['iata_code']}: {e}")
    time.sleep(1)

100%|██████████| 180/180 [12:12<00:00,  4.07s/it]


Saved 180 airports


In [ ]:
weather_df_part_1 = pd.read_parquet('../data/weather_data_part1.parquet')

In [12]:
weather_df_part_1.head()

,iata_code,datetime,precipitation,rain,snowfall,wind_speed,wind_gusts,temperature,weather_code,cloud_cover_low
0,ABE,2025-01-01T00:00,0.1,0.1,0.0,14.8,29.5,6.4,51,35
1,ABE,2025-01-01T01:00,0.4,0.4,0.0,14.9,31.0,7.6,51,34
2,ABE,2025-01-01T02:00,1.8,1.8,0.0,14.1,29.5,7.1,61,100
3,ABE,2025-01-01T03:00,3.0,3.0,0.0,8.8,28.4,6.9,63,100
4,ABE,2025-01-01T04:00,2.0,2.0,0.0,5.8,15.5,6.7,61,100


In [13]:
remaining_batch = airport_data.iloc[180:]

weather_data_2 = []

for _, row in tqdm(remaining_batch.iterrows(), total=len(remaining_batch)):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={row['latitude_deg']}&"
        f"longitude={row['longitude_deg']}&"
        f"start_date=2025-01-01&"
        f"end_date=2026-04-30&"
        f"hourly=precipitation,wind_speed_10m,snowfall,temperature_2m,rain,wind_gusts_10m,weather_code,cloud_cover_low"
    )
    
    try:
        response = requests.get(url, timeout=30).json()
        hourly = response['hourly']
        
        response_df = pd.DataFrame({
            'iata_code': row['iata_code'],
            'datetime': hourly['time'],
            'precipitation': hourly['precipitation'],
            'rain': hourly['rain'],
            'snowfall': hourly['snowfall'],
            'wind_speed': hourly['wind_speed_10m'],
            'wind_gusts': hourly['wind_gusts_10m'],
            'temperature': hourly['temperature_2m'],
            'weather_code': hourly['weather_code'],
            'cloud_cover_low': hourly['cloud_cover_low']
        })
        
        weather_data_2.append(response_df)
    except Exception as e:
        print(f"Failed for {row['iata_code']}: {e}")
    time.sleep(1)

weather_df_part2 = pd.concat(weather_data_2, ignore_index=True)
weather_df_part2.to_parquet('../data/weather_data_part2.parquet', index=False)
print(f"Saved {weather_df_part2['iata_code'].nunique()} airports")

100%|██████████| 177/177 [22:48<00:00,  7.73s/it]


Saved 177 airports


In [14]:
weather_df = pd.concat([weather_df_part_1, weather_df_part2], ignore_index=True)
weather_df.to_parquet('../data/weather_data.parquet', index=False)
print(f"Total: {weather_df['iata_code'].nunique()} airports")

Total: 357 airports


In [16]:
weather_df.head()

,iata_code,datetime,precipitation,rain,snowfall,wind_speed,wind_gusts,temperature,weather_code,cloud_cover_low
0,ABE,2025-01-01T00:00,0.1,0.1,0.0,14.8,29.5,6.4,51,35
1,ABE,2025-01-01T01:00,0.4,0.4,0.0,14.9,31.0,7.6,51,34
2,ABE,2025-01-01T02:00,1.8,1.8,0.0,14.1,29.5,7.1,61,100
3,ABE,2025-01-01T03:00,3.0,3.0,0.0,8.8,28.4,6.9,63,100
4,ABE,2025-01-01T04:00,2.0,2.0,0.0,5.8,15.5,6.7,61,100
